# Week 3, day 4 (morning) — Worksheet 10: Data quality checks, and validating the model

> *A dimensional model is useful only if it supports the questions business users
> need to answer.* — L03, slide 30

Two things close the day.

**Step 7 (slide 39)** is five check types against the loaded table:

| Check type | Example check |
|---|---|
| Row count | does `fact_enrollment` match the expected number of valid enrollment records? |
| Null | are required keys or measures missing? |
| Duplicate | was the same enrollment loaded more than once? |
| Referential integrity | does every fact row match valid dimension records? |
| Business rule | is `discount_amount <= tuition_amount`, and is `is_paid_in_full` calculated correctly? |

**Slide 30** is the other half, and it is the one people skip: take the six
business questions the model was designed for, and answer each one from the model.
A model that passes every technical check and cannot answer question one is a
failed model.

**Question 10 is supposed to raise an error** — and it is the most useful failure
in this folder.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 10 — Quality checks and validation. Run this once.
import pandas as pd

DATA = "data/"
load = lambda name: pd.read_csv(DATA + name + ".csv")

enr, tx = load("enrollment"), load("transaction")
crs, prg, coh = load("course"), load("program"), load("cohort")
stu, cat, dtype = load("students"), load("category"), load("discount_type")


def build_model():
    """Worksheets 06-09 in one function: the finished star schema."""
    test_ids = set(stu.loc[stu.stu_name.str.startswith("TEST"), "stu_id"])
    spine = enr[~enr.stu_id.isin(test_ids)].copy()

    def keymap(df, col, sk):
        out = df[[col]].drop_duplicates().sort_values(col).reset_index(drop=True)
        out[sk] = range(1, len(out) + 1)
        out = out.rename(columns={col: "src"})
        return pd.concat([pd.DataFrame([{"src": -1, sk: -1}]), out],
                         ignore_index=True)

    dupes = tx.drop(columns=["trans_id"]).duplicated()
    per = (tx[~dupes].groupby("enrl_id")
           .agg(tuition_amount=("full_price", "max"),
                amount_paid_to_date=("payment_amount", "sum"),
                dtid=("discount_type_id", "max")).reset_index())
    per = per.merge(dtype[["discount_type_id", "discount_amount"]],
                    left_on="dtid", right_on="discount_type_id", how="left")

    f = spine.merge(crs[["course_id", "program_id"]], on="course_id", how="left")
    f = f.merge(per, on="enrl_id", how="left")
    f["dtid"] = f.dtid.fillna(-1)
    for df, left, sk in [(crs, "course_id", "course_id_sk"),
                         (prg, "program_id", "program_id_sk"),
                         (coh, "cohort_id", "cohort_id_sk"),
                         (stu, "stu_id", "student_id_sk"),
                         (dtype, "discount_type_id", "promotion_id_sk")]:
        src_col = {"course_id_sk": "course_id", "program_id_sk": "program_id",
                   "cohort_id_sk": "cohort_id", "student_id_sk": "stu_id",
                   "promotion_id_sk": "discount_type_id"}[sk]
        f = f.merge(keymap(df, src_col, sk).rename(columns={"src": "_s"}),
                    left_on=left, right_on="_s", how="left").drop(columns=["_s"])
        f[sk] = f[sk].fillna(-1).astype(int)

    f["enrollment_date_id"] = pd.to_datetime(f.enrl_date).dt.strftime("%Y%m%d").astype(int)
    for c in ("tuition_amount", "discount_amount", "amount_paid_to_date"):
        f[c] = f[c].fillna(0.0)
    f["net_tuition_amount"] = f.tuition_amount - f.discount_amount
    f["enrollment_count"] = 1
    f["is_cancelled"] = (f.status == "cancelled").astype(int)
    f["is_paid_in_full"] = ((f.net_tuition_amount > 0)
                            & (f.amount_paid_to_date >= f.net_tuition_amount - 0.005)
                            ).astype(int)
    fact = f.rename(columns={"enrl_id": "enrollment_id",
                             "program_id_sk": "program_key",
                             "course_id_sk": "course_key",
                             "cohort_id_sk": "cohort_key",
                             "student_id_sk": "student_key",
                             "promotion_id_sk": "promotion_key"})
    return spine, fact[["enrollment_id", "program_key", "course_key",
                        "cohort_key", "student_key", "enrollment_date_id",
                        "promotion_key", "enrollment_count", "tuition_amount",
                        "discount_amount", "net_tuition_amount",
                        "amount_paid_to_date", "is_paid_in_full", "is_cancelled"]]


spine, fact = build_model()

# dim_course carries program_category, so slide 25's questions need one join.
dim_course = (crs[["course_id", "course_name", "hours"]]
              .merge(prg[["program_id", "program_name", "category_id"]],
                     left_on=crs.program_id, right_on="program_id", how="left")
              .merge(cat[["category_id", "category_name"]], on="category_id",
                     how="left"))
dim_course["category_name"] = dim_course["category_name"].fillna("Unknown")
dim_course = dim_course.sort_values("course_id").reset_index(drop=True)
dim_course.insert(0, "course_key", range(1, len(dim_course) + 1))

dim_date = pd.DataFrame({"full_date": pd.date_range(
    pd.to_datetime(enr.enrl_date).min(), pd.to_datetime(enr.enrl_date).max(),
    freq="D")})
dim_date["date_id"] = dim_date.full_date.dt.strftime("%Y%m%d").astype(int)
dim_date["month"] = dim_date.full_date.dt.to_period("M").astype(str)
dim_date["quarter"] = dim_date.full_date.dt.quarter
dim_date["year"] = dim_date.full_date.dt.year

print("fact_enrollment:", fact.shape)
print("dim_course:     ", dim_course.shape)
print("dim_date:       ", dim_date.shape)

PART A — slide 39's five checks

### Question 1

**Row count check.** Compare `fact_enrollment`'s row count with the expected count derived independently from the source, and print both plus the difference.
> **NOTE:** derive the expectation from the source, not from the fact table. A check that reads its own answer checks nothing.

In [ ]:
############################
## Your Code Here
############################

### Question 2

**Null check.** Print the null count for every column of `fact_enrollment`, and separately assert that the six key columns and five measures are complete.

In [ ]:
############################
## Your Code Here
############################

### Question 3

**Duplicate check.** Test three things: `enrollment_id` unique, no fully duplicated rows, and the grain from worksheet 02 — no repeat of `student_key + course_key + cohort_key`.
> **NOTE:** the third one is expected to find something. Worksheet 02 question 2 explains what.

In [ ]:
############################
## Your Code Here
############################

### Question 4

**Referential integrity check.** For each of the five surrogate keys, print how many fact rows reference a key that does not exist in the dimension, and how many sit on the `Unknown` member.

In [ ]:
############################
## Your Code Here
############################

### Question 5

**Business rule check.** Test slide 39's two rules — `discount_amount <= tuition_amount`, and `is_paid_in_full` recomputed from the measures — and print the violation count for each.

In [ ]:
############################
## Your Code Here
############################

### Question 6

Turn the five checks into one harness: a list of (name, expected, actual) tuples that prints a PASS/FAIL table and a final verdict.
> **NOTE:** a check suite that only prints numbers is a report. One that prints PASS/FAIL is a gate.

In [ ]:
############################
## Your Code Here
############################

PART B — slide 25's six business questions

### Question 7

**Q1 and Q2.** *How many students enrolled each day?* and *how is enrollment changing over time by course?* Answer both from the model, printing the three busiest days and enrollments by quarter for one course.

In [ ]:
############################
## Your Code Here
############################

### Question 8

**Q3 and Q4.** *Which courses have the highest enrollment?* and *which course-cohort groups have the highest full payment rate?* Answer both, and apply worksheet 03 question 7's lesson to the second.
> **NOTE:** a rate without its denominator is not an answer. Slide 30 says to benchmark; do the minimum version of that.

In [ ]:
############################
## Your Code Here
############################

### Question 9

**Q5 and Q6.** *Which course-cohort groups have the highest discount rates?* and *which may be over-discounted or underperforming?* Compute the discount rate as a ratio of sums, then find groups that are above the course benchmark on discount and below it on full payment.

In [ ]:
############################
## Your Code Here
############################

### Question 10

Finally, ask the model a seventh question: *which payment method do students who pay in full prefer?* Try `fact.groupby("payment_type_key")`. **This is supposed to fail.** Say what the failure means for the model.
> **NOTE:** worksheet 07 question 2 already told you why this column cannot exist on this table.

In [ ]:
############################
## Your Code Here
############################